<div style="background-color: #ffffff; color: #000000; padding: 30px;">
<img src="../media/images/kisz_logo.png" width="192" height="69" align="right" style="margin-right: 50px; margin-bottom: 50px;">
<h1>Time Series Analysis and Forecasting</h1>
</div>

<div style="background-color: #f6a800; color: #ffffff; padding: 10px;">
<h2>Solutions</h2>
<h2>Notebook B02: ARIMA Models</h2>
</div>

Worked solutions to the 3 exercises in
[Notebook B02: ARIMA Models](../notebooks/B02_ARIMA_models.ipynb).

**Try each exercise yourself first.** These notebooks are most useful as a check on your reasoning, and
least useful as something to read straight through. An exercise you attempted and got wrong teaches more
than a solution you agreed with.

Where an exercise asks a question rather than requesting code, the answer is written out under the code
that produces it. Several of them have answers that are more interesting than they look.

The setup cell below reproduces the state the exercises assume, so this notebook runs on its own.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="setup">Setup</h3>
</div>

In [ ]:
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import acf, adfuller, kpss

sys.path.append("../notebooks")
import nb_config

sns.set_theme(style="whitegrid")

series = pd.read_parquet(nb_config.CDC_TEMP_PATH)["Brandenburg/Berlin"].asfreq("MS")

TEST_MONTHS = 24
SEASON_LENGTH = 12
train, test = series.iloc[:-TEST_MONTHS], series.iloc[-TEST_MONTHS:]


def check_stationarity(values, label):
    values = pd.Series(values).dropna()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        adf_p = adfuller(values, autolag="AIC")[1]
        kpss_p = kpss(values, regression="c", nlags="auto")[1]
    return {
        "series": label,
        "ADF p": adf_p,
        "ADF says": "stationary" if adf_p < 0.05 else "NOT stationary",
        "KPSS p": kpss_p,
        "KPSS says": "NOT stationary" if kpss_p < 0.05 else "stationary",
    }


print(f"Train {len(train)} months")

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="exercise-1">Exercise 1</h3>
</div>

> Apply an ordinary first difference (`train.diff()`) instead of a seasonal one and run the tests again. Do they agree? Plot the result: what has first differencing done to the seasonal pattern, and why is it the wrong tool here?

In [ ]:
variants = {
    "raw": train,
    "diff(1) — ordinary": train.diff(),
    "diff(12) — seasonal": train.diff(SEASON_LENGTH),
}

tests = pd.DataFrame([check_stationarity(values, name) for name, values in variants.items()])
tests.set_index("series").round(4)

**Yes, the tests agree — and that is exactly the problem.**

On the raw series ADF and KPSS disagree, as the notebook showed. After an ordinary first difference **both
report stationarity**, just as they do after a seasonal difference. On the evidence of these two tests
alone, `diff(1)` has done its job.

It has not. Look at what happened to the seasonal structure.

In [ ]:
comparison = pd.DataFrame({
    name: {
        "std": values.std(),
        "ACF at lag 12": acf(values.dropna(), nlags=13)[12],
        "ACF at lag 1": acf(values.dropna(), nlags=13)[1],
    }
    for name, values in variants.items()
}).T

comparison.round(3)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 7))

axes[0, 0].plot(train["2015":], color="steelblue", linewidth=1.0)
axes[0, 0].set_title("Raw", fontsize=12, fontweight="bold")
axes[0, 1].plot(train.diff()["2015":], color="crimson", linewidth=1.0)
axes[0, 1].set_title("After diff(1)", fontsize=12, fontweight="bold")

plot_acf(train.diff().dropna(), lags=30, ax=axes[1, 0], color="crimson",
         vlines_kwargs={"colors": "crimson"})
axes[1, 0].set_title("ACF after diff(1)", fontsize=12, fontweight="bold")

plot_acf(train.diff(SEASON_LENGTH).dropna(), lags=30, ax=axes[1, 1], color="seagreen",
         vlines_kwargs={"colors": "seagreen"})
axes[1, 1].set_title("ACF after diff(12)", fontsize=12, fontweight="bold")

for ax in axes.ravel():
    ax.grid(axis="y", linestyle="--", alpha=0.4)
for ax in axes[1]:
    ax.set_ylim(-0.8, 0.8)
    ax.set_xlabel("Lag (months)")

plt.tight_layout()
plt.show()

**First differencing has left the seasonality almost entirely intact.** The ACF at lag 12 is **+0.674**
after `diff(1)`, against +0.920 on the raw series — reduced, but still enormous. The seasonal difference
takes it to -0.501, which is a different kind of number entirely: a single spike created by the
differencing operation itself rather than a surviving cycle.

The reason is what each operation actually computes. `diff(1)` subtracts last month from this month, so it
measures the **month-to-month change**. On a seasonal series that change is itself seasonal: the jump from
January to February is reliably different from the jump from July to August. Differencing at lag 1 removes
a trend, which this series barely has, and does almost nothing about a twelve-month cycle.

`diff(12)` subtracts the same month a year earlier, which is the operation that matches the structure.

Two lessons worth carrying forward.

**Match the differencing lag to the period you are removing.** The choice is not "how many differences"
but "differences at which lag", and the answer comes from looking at the data rather than from a default.

**And passing a stationarity test is not the same as being ready to model.** Both ADF and KPSS are
satisfied by the first-differenced series, yet its dominant feature is a correlation of 0.67 at lag 12 that
no ARMA model with small orders will capture. The tests answer a narrow question about unit roots and
constant behaviour; they do not check that you have removed the structure you meant to remove. **Always
look at the ACF as well.**

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="exercise-2">Exercise 2</h3>
</div>

> Run the diagnostics from Notebook A06 on the chosen model's residuals (`best.resid`): plot the ACF and run the Ljung-Box test. Compare with what the seasonal naive residuals looked like. Has SARIMA extracted the structure the baseline left behind?

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    best = ARIMA(train, order=(2, 0, 1), seasonal_order=(0, 1, 1, SEASON_LENGTH)).fit()

# Drop the first cycle: differencing leaves the initial residuals undefined
sarima_residuals = best.resid.dropna().iloc[SEASON_LENGTH + 1:]
seasonal_naive_residuals = (series - series.shift(SEASON_LENGTH)).dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

for ax, (name, residuals) in zip(
    axes,
    [("Seasonal naive", seasonal_naive_residuals), ("SARIMA(2,0,1)(0,1,1,12)", sarima_residuals)],
):
    plot_acf(residuals, lags=36, ax=ax, color="steelblue", vlines_kwargs={"colors": "steelblue"})
    ax.set_title(f"{name} residuals", fontsize=13, fontweight="bold")
    ax.set_xlabel("Lag (months)")
    ax.set_ylim(-0.7, 0.7)
    ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
summary = []
for name, residuals in [("Seasonal naive", seasonal_naive_residuals),
                        ("SARIMA", sarima_residuals)]:
    autocorrelation = acf(residuals, nlags=24)
    ljung_box = acorr_ljungbox(residuals, lags=[12, 24], return_df=True)
    summary.append({
        "model": name,
        "residual std": residuals.std(),
        "ACF lag 1": autocorrelation[1],
        "ACF lag 12": autocorrelation[12],
        "Ljung-Box p (12)": ljung_box["lb_pvalue"].iloc[0],
        "Ljung-Box p (24)": ljung_box["lb_pvalue"].iloc[1],
    })

pd.DataFrame(summary).set_index("model")

**Yes, completely — and this is the cleanest diagnostic result in the course.**

The seasonal naive residuals carry a correlation of +0.21 at lag 1 and -0.50 at lag 12, and Ljung-Box
rejects independence at p ≈ 1e-109. The SARIMA residuals carry **-0.017 at lag 1 and -0.004 at lag 12**,
and Ljung-Box returns **p ≈ 0.80**.

A p-value of 0.80 means the test finds no evidence of remaining autocorrelation at all. The residuals are
indistinguishable from white noise, which is the condition Notebook
[A06](../notebooks/A06_Evaluating_models.ipynb) described as the point where further modelling stops
paying. The residual standard deviation has fallen from 2.69 to 1.90 at the same time.

It is worth seeing how precisely the model matched the diagnosis. The baseline's residuals showed exactly
two features: short-range persistence at lag 1, and a spike at the seasonal lag. The chosen model is
SARIMA(2,0,1)(0,1,1,12) — autoregressive terms for the first, a seasonal moving-average term for the
second. The ACF plot said what was missing and the search found a model that supplies it.

Compare this with the same exercise in Notebook A06, where Holt-Winters removed the seasonal correlation
but left +0.20 at lag 1 and still rejected at p ≈ 1e-12. That leftover is precisely what the AR terms here
absorb. The two notebooks, read together, show a diagnostic pointing at a gap and a model closing it.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="exercise-3">Exercise 3</h3>
</div>

> Add the day-of-week as exogenous dummy variables (`pd.get_dummies(store.index.dayofweek)`) and drop the seasonal order to `(0, 0, 0, 0)`. Does explicit calendar information do the same job as the weekly seasonal terms, and which gives the better AIC?

In [ ]:
sales = pd.read_csv(nb_config.ROSSMANN_TRAIN_PATH, parse_dates=["Date"], low_memory=False)
store = sales[sales["Store"] == 1].set_index("Date").sort_index().asfreq("D")

FORECAST_DAYS = 28
target = store["Sales"].astype(float)
target_train, target_test = target.iloc[:-FORECAST_DAYS], target.iloc[-FORECAST_DAYS:]

# Day-of-week dummies, dropping one level to avoid perfect collinearity with the constant
weekday_dummies = pd.get_dummies(store.index.dayofweek, prefix="dow", drop_first=True).astype(float)
weekday_dummies.index = store.index

base_exog = store[["Open", "Promo"]].astype(float)
combined_exog = pd.concat([base_exog, weekday_dummies], axis=1)

print(f"{weekday_dummies.shape[1]} day-of-week dummies + {base_exog.shape[1]} other columns")

In [ ]:
def mean_absolute_error(actual, forecast):
    return float(np.mean(np.abs(np.asarray(actual) - np.asarray(forecast))))


def fit_and_score(exog, order, seasonal_order, label):
    """Fit one specification and report AIC alongside held-out error."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        model = ARIMA(
            target_train,
            exog=exog.iloc[:-FORECAST_DAYS] if exog is not None else None,
            order=order,
            seasonal_order=seasonal_order,
        ).fit()

    forecast = model.forecast(
        FORECAST_DAYS, exog=exog.iloc[-FORECAST_DAYS:] if exog is not None else None
    )

    return {
        "specification": label,
        "parameters": len(model.params),
        "AIC": model.aic,
        "Test MAE": mean_absolute_error(target_test, forecast),
    }


results = [
    fit_and_score(base_exog, (1, 0, 1), (1, 0, 1, 7), "weekly seasonal terms"),
    fit_and_score(combined_exog, (1, 0, 1), (0, 0, 0, 0), "day-of-week dummies"),
    fit_and_score(combined_exog, (1, 0, 1), (1, 0, 1, 7), "both"),
    fit_and_score(base_exog, (1, 0, 1), (0, 0, 0, 0), "neither"),
]

pd.DataFrame(results).set_index("specification").round(1)

**Yes, the dummies do the same job — and the two columns of that table disagree about how well.**

Start with what both agree on. Either encoding of the weekly rhythm is far better than none: dropping both
costs 215 AIC points and raises held-out error from 266 to 456. The weekly cycle is the dominant structure
in this series, and it has to be modelled somehow.

Then the disagreement:

| | AIC prefers | Held-out MAE prefers |
|---|---|---|
| Best | both (14318) | weekly seasonal terms (266) |
| Then | dummies (14346) | both (285) |
| Then | seasonal terms (14393) | dummies (377) |

**AIC ranks the dummies above the seasonal terms. Held-out error ranks them well below.** Both cannot be
right about which model to ship.

The two encodings differ in a way that explains it. The seasonal terms `(1,0,1,7)` model the weekly
pattern *dynamically*, estimating it from recent data and letting it evolve. The dummies model it
*deterministically*: six fixed coefficients, constant across two and a half years. The deterministic
version fits the training period slightly better per parameter, which is what AIC measures, and it cannot
adapt when the pattern shifts — which is what the held-out period rewards.

This is the caution from section 6 of the notebook arriving in a concrete case. **AIC measures fit to the
data the model was trained on, with a penalty for complexity. It does not measure whether the model will
generalise**, and here it prefers the specification that generalises worse. Use it to narrow a field;
confirm on held-out data before choosing.

One technical note: **`drop_first=True` is not optional.** Seven dummies plus a constant are perfectly
collinear, since the dummies sum to one on every row, and the model cannot identify all eight
coefficients. Dropping one level leaves six differences from a reference day, which is the more readable
parameterisation anyway.

---

Back to [Notebook B02](../notebooks/B02_ARIMA_models.ipynb), or on to
[Notebook B03](../notebooks/B03_Advanced_statistical_models.ipynb).